# Lesson 5.2 — Divergence as a Signal

Two channels: STATE divergence (snapshot vs report, staleness) and OUTCOME divergence (predicted vs actual harvest, surprising behaviour).

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception, understand,
                                          run_pipeline)
from modules.module10.twin import (DigitalTwin, GroundTruth, outcome_gap,
                                   monitor, predict, compare_futures)
def victim_and_xy(seed=1):
    w = GreenhouseWorld.demo_row(n=6, seed=seed)
    dets = model_perception(w, rng=np.random.default_rng(7))
    v = understand(dets, w)[1]['id']
    vxy = next(d['xy'] for d in dets if d['id']==v)
    return v, vxy
checks = []
v, vxy = victim_and_xy()
# STATE channel: move reality's state -> state divergence rises
g1 = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1))
t1 = DigitalTwin(g1.world); t1.sync(g1.report())
g1.world.q = g1.world.q + np.array([0.08, 0.04])
state = t1.divergence(g1.report())
print('STATE channel -> q_gap=%.3f (drift / stale snapshot)' % state['q_gap'])
checks.append(state['q_gap'] > 0)
# OUTCOME channel: unmodeled effect -> outcome divergence rises
g2 = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1), unmodeled={v: dict(obstacle=(vxy,0.25))})
t2 = DigitalTwin(g2.world)
out = outcome_gap(t2.simulate(), g2.run())
print('OUTCOME channel -> n_diffs=%d on %s (surprising behaviour)' % (out['n_outcome_diffs'], out['harvested_only_in_sim']))
checks.append(out['n_outcome_diffs'] > 0 and v in out['harvested_only_in_sim'])
# the two channels are distinct: a state move does NOT imply an outcome surprise
clean_out = outcome_gap(t1.simulate(), GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1)).run())
checks.append(clean_out['match'] is True)
print('state-only move leaves outcome match intact:', clean_out['match'])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


STATE channel -> q_gap=0.089 (drift / stale snapshot)


OUTCOME channel -> n_diffs=2 on ['F3'] (surprising behaviour)


state-only move leaves outcome match intact: True
3/3 checks passed.
All checks passed.
